# FLIP — 손글씨 답 인식 YOLO 학습 (Colab)

`synth_data.py` 업로드 → xainano 다운로드(+data.rar 추출) → 합성 → yolov8n 학습 → **성능 기록** → `best.pt` 다운로드.

**먼저:** 상단 메뉴 `런타임 > 런타임 유형 변경 > GPU(T4)` 로 GPU 켜기.

In [ ]:
# 1. 설치
!pip -q install ultralytics kagglehub

## 2. synth_data.py 생성 (노트북에 내장 — 업로드 불필요)
아래 셀이 최신 `synth_data.py`를 그대로 써넣는다.


In [ ]:
%%writefile synth_data.py
"""
FLIP 답안 인식용 YOLO-det 학습 데이터 합성기.

단일-문자 손글씨 데이터셋(xainano 등, 클래스별 폴더)에서 글자를 꺼내
캔버스에 왼->오로 붙여 "답안 문자열" 이미지를 만들고,
붙인 위치가 곧 정답 박스라서 YOLO 라벨을 0초에 자동 생성한다.

실행:
  python synth_data.py --selftest                # 데이터셋 없이 로직 검증
  python synth_data.py --dataset <xainano_dir> --out data --n 4000

xainano 폴더명이 아래 CLASS_FOLDERS와 다르면 그 dict만 고치면 됨.
"""
import argparse
import os
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter

# 우리가 쓸 클래스 (id = 인덱스). 정답에 나오는 것만.
CLASSES = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "-", "/", "=", "pi"]

# 우리 클래스 -> xainano 하위 폴더명.  다운로드한 폴더 보고 안 맞으면 여기만 수정.
CLASS_FOLDERS = {
    "0": "0", "1": "1", "2": "2", "3": "3", "4": "4",
    "5": "5", "6": "6", "7": "7", "8": "8", "9": "9",
    "-": "-",              # 음수/마이너스
    "/": "forward_slash",  # 분수 슬래시 (xainano에서 div 로 되어 있으면 "div" 로 변경)
    "=": "=",
    "pi": "pi",
}

CANVAS_H = 64          # 캔버스 높이(px)
GLYPH_H_RANGE = (30, 48)  # 글자 높이 랜덤 범위
# 글자 간격(px). 샘플마다 한 모드를 골라 그 답 전체에 일관 적용
# (사람은 한 답을 대체로 같은 간격으로 씀). 둘 다 학습해야 함.
GAP_TIGHT = (-12, -2)   # 붙여쓰기/겹침 (예: 붙은 -3, 2/5)
GAP_SPACED = (2, 14)    # 띄어쓰기/단독 (예: 벌어진 365)
PAD = 8                # 좌우/상하 여백
INK_THRESH = 40        # alpha>이 값이면 잉크로 간주 (박스 계산용)


def normalize_glyph(img: Image.Image) -> np.ndarray:
    """어떤 극성이든 '흰 배경 위 검은 잉크'로 통일해 alpha(잉크 강도) 배열 반환."""
    a = np.asarray(img.convert("L"), dtype=np.float32)
    if a.mean() < 127:          # 검은 배경 위 흰 글씨면 반전
        a = 255.0 - a
    ink = 255.0 - a             # 검을수록 잉크 강함
    ink[ink < 0] = 0
    return ink                  # HxW, 0..255


def ink_bbox(ink: np.ndarray):
    """잉크 영역의 타이트 박스 (x1,y1,x2,y2). 없으면 None."""
    ys, xs = np.where(ink > INK_THRESH)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def load_pool(dataset_dir: str):
    """클래스 -> 이미지 파일 경로 리스트."""
    pool = {}
    for cls in CLASSES:
        folder = Path(dataset_dir) / CLASS_FOLDERS[cls]
        if not folder.is_dir():
            raise FileNotFoundError(
                f"클래스 '{cls}' 폴더 없음: {folder}  -> CLASS_FOLDERS 확인/수정"
            )
        files = sorted(p for p in folder.iterdir()   # sorted: run마다 순서 고정(재현성)
                       if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"))
        if not files:
            raise FileNotFoundError(f"{folder} 에 이미지가 없음")
        pool[cls] = files
    return pool


def fake_pool(tmp: Path):
    """셀프테스트용: 폰트로 글리프를 그려 가짜 데이터셋 생성(데이터셋 불필요)."""
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial.ttf", 40)
    except OSError:
        font = ImageFont.load_default()
    glyph_text = {**{c: c for c in "0123456789"}, "-": "-", "/": "/", "=": "=", "pi": "n"}
    pool = {}
    for cls in CLASSES:
        d = tmp / CLASS_FOLDERS[cls]
        d.mkdir(parents=True, exist_ok=True)
        paths = []
        for i in range(6):
            im = Image.new("L", (45, 45), 255)
            ImageDraw.Draw(im).text((12, 2), glyph_text[cls], fill=0, font=font)
            p = d / f"{i}.png"
            im.save(p)
            paths.append(p)
        pool[cls] = paths
    return pool


def compose_sequence():
    """그럴듯한 답 시퀀스(클래스 라벨 리스트). 모든 클래스가 골고루 나오게."""
    r = random.random()
    if r < 0.45:                                   # 정수 (부호 가능)
        seq = random.choices("0123456789", k=random.randint(1, 3))
        if random.random() < 0.3:
            seq = ["-"] + seq
    elif r < 0.75:                                 # 분수
        num = random.choices("0123456789", k=random.randint(1, 2))
        den = random.choices("0123456789", k=random.randint(1, 2))
        seq = num + ["/"] + den
        if random.random() < 0.3:
            seq = ["-"] + seq
    elif r < 0.9:                                  # pi 포함
        seq = random.choices("0123456789", k=random.randint(0, 2)) + ["pi"]
        if random.random() < 0.4:
            seq += ["/"] + random.choices("0123456789", k=1)
    else:                                          # '=' 노출용 (희소)
        seq = random.choices("0123456789", k=1) + ["="] + \
              random.choices("0123456789", k=random.randint(1, 2))
    return seq


def render_sample(pool):
    """한 장 합성. return (PIL RGB image, [(class_id, x1,y1,x2,y2), ...])."""
    seq = compose_sequence()
    glyphs = []
    for cls in seq:
        ink = normalize_glyph(Image.open(random.choice(pool[cls])))
        h = random.randint(*GLYPH_H_RANGE)
        w = max(1, int(ink.shape[1] * h / ink.shape[0]))
        g = np.asarray(Image.fromarray(ink).resize((w, h)), dtype=np.float32)
        ang = random.uniform(-12, 12)
        g = np.asarray(Image.fromarray(g).rotate(ang, expand=True, resample=Image.BILINEAR),
                       dtype=np.float32)
        glyphs.append((CLASSES.index(cls), g))

    gap = GAP_TIGHT if random.random() < 0.5 else GAP_SPACED  # 이 답의 간격 모드
    total_w = PAD * 2 + sum(g.shape[1] for _, g in glyphs) + \
        gap[1] * max(0, len(glyphs) - 1)
    canvas = np.zeros((CANVAS_H, total_w), dtype=np.float32)  # 잉크 누적(검을수록 큼)
    boxes = []
    x = PAD
    for cid, g in glyphs:
        gh, gw = g.shape
        y = random.randint(PAD, max(PAD, CANVAS_H - gh - PAD))
        # 캔버스 밖으로 삐져나가면 잘라서 붙임(회전으로 커진 경우 대비)
        yh, xw = min(gh, CANVAS_H - y), min(gw, canvas.shape[1] - x)
        sub = g[:yh, :xw]
        canvas[y:y + yh, x:x + xw] = np.maximum(canvas[y:y + yh, x:x + xw], sub)
        bb = ink_bbox(sub)  # 실제로 붙은 부분에서 박스 계산
        if bb:  # 캔버스 좌표로 이동
            bx1, by1, bx2, by2 = bb
            boxes.append((cid, x + bx1, y + by1, x + bx2, y + by2))
        x += gw + random.randint(*gap)

    img = Image.fromarray(255.0 - canvas).convert("RGB")  # 흰 배경 위 검은 잉크
    # 촬영처럼 가벼운 노이즈/블러/명암
    if random.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 0.8)))
    arr = np.asarray(img, dtype=np.float32)
    arr += np.random.normal(0, random.uniform(2, 9), arr.shape)
    arr = np.clip(arr * random.uniform(0.85, 1.0) + random.uniform(0, 20), 0, 255)
    return Image.fromarray(arr.astype(np.uint8)), boxes


def to_yolo(box, W, H):
    _, x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
    bw, bh = (x2 - x1) / W, (y2 - y1) / H
    return box[0], cx, cy, bw, bh


def write_split(pool, out: Path, n, split):
    (out / "images" / split).mkdir(parents=True, exist_ok=True)
    (out / "labels" / split).mkdir(parents=True, exist_ok=True)
    for i in range(n):
        img, boxes = render_sample(pool)
        if not boxes:
            continue
        W, H = img.size
        img.save(out / "images" / split / f"{i:06d}.jpg", quality=90)
        lines = []
        for b in boxes:
            cid, cx, cy, bw, bh = to_yolo(b, W, H)
            lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        (out / "labels" / split / f"{i:06d}.txt").write_text("\n".join(lines))


def write_yaml(out: Path):
    names = "\n".join(f'  {i}: "{c}"' for i, c in enumerate(CLASSES))  # "-" "/" "=" 는 따옴표 필수
    (out / "data.yaml").write_text(
        f"path: {out.resolve()}\ntrain: images/train\nval: images/val\n"
        f"nc: {len(CLASSES)}\nnames:\n{names}\n"
    )


def selftest():
    import tempfile
    random.seed(0)
    np.random.seed(0)
    with tempfile.TemporaryDirectory() as td:
        pool = fake_pool(Path(td))
        for _ in range(200):
            img, boxes = render_sample(pool)
            W, H = img.size
            assert len(boxes) >= 1, "박스가 하나도 안 생김"
            for b in boxes:
                cid, cx, cy, bw, bh = to_yolo(b, W, H)
                assert 0 <= cid < len(CLASSES)
                assert 0.0 < cx < 1.0 and 0.0 < cy < 1.0, "중심이 이미지 밖"
                assert 0.0 < bw <= 1.0 and 0.0 < bh <= 1.0, "박스 크기 이상"
                _, x1, y1, x2, y2 = b
                assert 0 <= x1 < x2 <= W and 0 <= y1 < y2 <= H, "박스가 이미지 밖"
    print("selftest OK: 라벨 지오메트리 정상")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", help="xainano 클래스별 폴더 경로")
    ap.add_argument("--out", default="data")
    ap.add_argument("--n", type=int, default=4000, help="train 장수 (val=10%)")
    ap.add_argument("--selftest", action="store_true")
    args = ap.parse_args()

    if args.selftest:
        selftest()
        return
    if not args.dataset:
        ap.error("--dataset 또는 --selftest 필요")

    pool = load_pool(args.dataset)
    out = Path(args.out)
    write_split(pool, out, args.n, "train")
    write_split(pool, out, max(1, args.n // 10), "val")
    write_yaml(out)
    print(f"완료: {out}/  (train={args.n}, val={args.n // 10})")
    print(f"학습:  yolo detect train model=yolov8n.pt data={out}/data.yaml imgsz=128 epochs=60")


if __name__ == "__main__":
    main()


## 3. xainano 다운로드 + data.rar 추출 + 클래스 루트 찾기
⚠️ 진짜 데이터는 **`data.rar`** 안에 있고 `extracted_images/`는 일부(숫자 0까지)만 풀려있다.

In [ ]:
import kagglehub, os, glob, subprocess
path = kagglehub.dataset_download("xainano/handwrittenmathsymbols")
print("downloaded:", path)

rar = glob.glob(os.path.join(path, "**", "data.rar"), recursive=True)
if rar:
    print("data.rar 발견 -> 추출:", rar[0])
    subprocess.run(["apt-get", "-qq", "install", "-y", "unar"], check=True)
    dest = "/content/xai"; os.makedirs(dest, exist_ok=True)
    subprocess.run(["unar", "-q", "-f", "-o", dest, rar[0]], check=True)
    search_base = dest
else:
    search_base = path

root = None
for d, subs, _ in os.walk(search_base):
    if all(os.path.isdir(os.path.join(d, x)) for x in ["0", "1", "2"]):
        root = d; break
assert root, "클래스 폴더 루트를 못 찾음"
print("class root:", root)
print("folders:", sorted(os.listdir(root)))

## 4. 폴더명 매핑 확인 (`/`=forward_slash 등)

In [ ]:
import synth_data as S

def pick(cands):
    for c in cands:
        if os.path.isdir(os.path.join(root, c)): return c
    return None

S.CLASS_FOLDERS.update({
    **{str(i): str(i) for i in range(10)},
    "-":  pick(["-", "minus", "dash", "sub"]),
    "/":  pick(["forward_slash", "div", "slash", "frac"]),
    "=":  pick(["=", "eq", "equal", "equals"]),
    "pi": pick(["pi", "Pi", "PI"]),
})
mapping = {k: S.CLASS_FOLDERS[k] for k in ["-", "/", "=", "pi"]}
print("mapping:", mapping)
assert all(mapping.values()), f"못 찾은 클래스: {mapping}"
print("OK")

## 5. 합성

In [ ]:
import random, numpy as np
from pathlib import Path
random.seed(0); np.random.seed(0)
pool = S.load_pool(root)
out = Path("data")
S.write_split(pool, out, 4000, "train")
S.write_split(pool, out, 400,  "val")
S.write_yaml(out)
print("생성됨:", os.listdir("data"))

## 6. 학습 (yolo26n)


In [ ]:
!yolo detect train model=yolo26n.pt data=data/data.yaml imgsz=128 epochs=60 patience=15 \
  fliplr=0.0 mosaic=0.0
# fliplr=0: 좌우반전 끔(2/5->5/2 방지, 숫자 깨짐 방지)  mosaic=0: 추론은 답 하나 크롭이라 콜라주 불필요

## 7. 성능 기록  ⭐
- **문자열 완전일치율** = 답을 통째로 맞게 읽은 비율 (FLIP에서 진짜 중요한 지표)
- **mAP** = 박스 검출 정확도 (참고)
- 학습 곡선·혼동행렬 표시, `metrics.txt` 저장

In [ ]:
import glob, os
from ultralytics import YOLO
m = YOLO("runs/detect/train/weights/best.pt")

# (1) 박스 단위 mAP
mv = m.val(data="data/data.yaml", imgsz=128, verbose=False)
map50, map5095 = float(mv.box.map50), float(mv.box.map)

# (2) 문자열 단위 일치율
def gt_string(lab):
    rows = []
    for line in open(lab).read().splitlines():
        if line.strip():
            c, cx, *_ = line.split(); rows.append((float(cx), S.CLASSES[int(c)]))
    return "".join(ch for _, ch in sorted(rows))

def pred_string(img):
    r = m.predict(img, imgsz=128, conf=0.25, agnostic_nms=False, verbose=False)[0]
    dets = sorted((float(b.xywh[0][0]), S.CLASSES[int(b.cls)]) for b in r.boxes)
    return "".join(ch for _, ch in dets)

exact = total = charhit = charall = 0
for im in sorted(glob.glob("data/images/val/*.jpg")):
    lab = im.replace("/images/", "/labels/")[:-4] + ".txt"
    if not os.path.exists(lab): continue
    gt, pr = gt_string(lab), pred_string(im)
    total += 1; exact += (gt == pr)
    for a, b in zip(gt, pr): charhit += (a == b)
    charall += max(len(gt), len(pr))

summary = (f"[FLIP 학습 결과]\n"
           f"문자열 완전일치율: {exact/total:.1%}  ({exact}/{total})\n"
           f"글자 정확도(근사) : {charhit/max(1,charall):.1%}\n"
           f"mAP50            : {map50:.3f}\n"
           f"mAP50-95         : {map5095:.3f}\n"
           f"설정: yolov8n, imgsz=128, epochs=60, train=4000/val=400")
print(summary)
open("metrics.txt", "w").write(summary)

from IPython.display import Image as IPy, display
for f in ["results.png", "confusion_matrix.png"]:
    p = os.path.join("runs/detect/train", f)
    if os.path.exists(p): print(f); display(IPy(p))

## 8. 다운로드 (best.pt + metrics.txt)
`metrics.txt` 내용은 `docs/training-log.md` 에 붙여넣어 기록.
로컬 테스트: `python infer.py --model best.pt --image crop.jpg --answer -3/4`

In [ ]:
from google.colab import files
files.download("runs/detect/train/weights/best.pt")
files.download("metrics.txt")